In [6]:
!pip install pandas -q
!pip install duckdb -q

In [7]:
import pandas as pd
import duckdb as duck

In [8]:
data_dir = "../csv"  
sales = pd.read_csv(f"{data_dir}/sales.csv", parse_dates=['DATE'])
parts = pd.read_csv(f"{data_dir}/parts.csv")
parts['SQFT'] = parts['SQFT'].astype('Int64')
f"rows - parts: {len(parts)}, sales: {len(sales)}"

'rows - parts: 7, sales: 84'

In [9]:
joined = duck.query("""
    SELECT s.*, p.prod, p.part_type, p.sqft
    FROM sales s
    INNER JOIN parts p ON s.part_id = p.part_id""")

joined
full = joined.df()
full

,DATE,SUPPLIER,LOCALE,LICENSEE,PO,PART_ID,QTY,AMT,PROD,PART_TYPE,SQFT
0,2024-01-10,nippon-metal,osaka,ninja-roofing,PO24001,tin-clip,55,13750,tin-roof,None,10
1,2024-01-18,nippon-metal,tokyo,rice-roofers,PO24002,tin-decoration,40,16000,tin-roof,None,<NA>
2,2024-01-25,us-steel,denver,ninja-roofing,PO24003,glass-clip,45,22500,glass-roof,new,100
3,2024-01-28,us-steel,starbase,global-roof,PO24004,glass-decoration,35,28000,glass-roof,new,<NA>
4,2024-02-08,nippon-metal,osaka,global-roof,PO24005,roof-polish,60,9000,None,None,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
79,2025-11-20,us-steel,starbase,global-roof,PO25037,glass-clip,65,32500,glass-roof,new,100
80,2025-12-08,nippon-metal,tokyo,zztop-roof,PO25038,tin-clip,75,18750,tin-roof,None,10
81,2025-12-15,nippon-metal,osaka,ranger-roofing,PO25039,tin-decoration,45,18000,tin-roof,None,<NA>
82,2025-12-22,us-steel,denver,ninja-roofing,PO25040,glass-decoration,60,48000,glass-roof,new,<NA>


In [10]:
duck.query("""
    DROP VIEW IF EXISTS sales_by_month;
    
    CREATE VIEW sales_by_month AS
    SELECT 
        strftime('%Y-%m', date) AS month,
        supplier,
        licensee,
        CAST(SUM(qty) AS INTEGER) AS tot_qty,
        CAST(SUM(amt) AS INTEGER) AS tot_amt
    FROM sales
    GROUP BY month, supplier, licensee
""")

duck.query("SELECT * FROM sales_by_month ORDER BY month, supplier, licensee").df()

,month,SUPPLIER,LICENSEE,tot_qty,tot_amt
0,2024-01,nippon-metal,ninja-roofing,55,13750
1,2024-01,nippon-metal,rice-roofers,40,16000
2,2024-01,us-steel,global-roof,35,28000
3,2024-01,us-steel,ninja-roofing,45,22500
4,2024-02,nippon-metal,global-roof,60,9000
...,...,...,...,...,...
78,2025-11,us-steel,rice-roofers,30,4500
79,2025-12,nippon-metal,ranger-roofing,45,18000
80,2025-12,nippon-metal,zztop-roof,75,18750
81,2025-12,us-steel,global-roof,35,5250
